# MVS-XAI: Multi-View Stacking Ensemble with Explainable AI
## IEEE-CIS Pipeline (590K E-Commerce Fraud Records)
---
**Full 8-Stage Architecture** as defined in the IEEE Proposal Figure 1.

| Stage | Component | Status |
|-------|-----------|--------|
| 1 | Preprocessing (Null, Encoding, Min-Max, FE) | ✅ |
| 2 | Walk-Forward CV (5 Blocks) + K-Means SMOTE | ✅ |
| 3 | Multi-View: Tabular + Sequential (T=10) + Graph (2-hop ego) | ✅ |
| 4 | 5 Base Models: RF, XGB, LGB, LSTM, GAT + OOF | ✅ |
| 5 | Meta-Learner: Logistic Regression (L2) on OOF [Nx5] | ✅ |
| 6 | 4-Level XAI: SHAP, LIME, DiCE, Anchors | ✅ |
| 7 | Dual-Output: Real-Time + Audit Report | ✅ |
| 8 | HITL Escalation + Regulatory Mapping | ✅ |

In [ ]:
# ====== MOUNT GOOGLE DRIVE ======
from google.colab import drive
drive.mount('/content/drive')

### 📥 Hướng dẫn tải Dataset IEEE-CIS
Nếu chưa có dataset trên Drive, chạy cell bên dưới để tải từ Kaggle trực tiếp vào Drive.

In [ ]:
# ====== [OPTIONAL] DOWNLOAD DATASET FROM KAGGLE ======
# Chỉ chạy cell này nếu chưa có file trên Drive.
# Bước 1: Upload file kaggle.json (API key) lên Colab.
#   - Vào https://www.kaggle.com/settings -> Create New Token -> Tải kaggle.json
#   - Upload lên Colab qua sidebar Files
#
# Bước 2: Chạy cell này:

import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'
# !pip install -q kaggle
# !kaggle competitions download -c ieee-fraud-detection -p /content/drive/MyDrive/ieee-cis-fraud-detection/
# !cd /content/drive/MyDrive/ieee-cis-fraud-detection/ && unzip -o ieee-fraud-detection.zip
print('Cell này chỉ chạy khi cần download. Bỏ comment các dòng trên để tải dataset.')

In [ ]:
# ====== DEPENDENCY INSTALLATION ======
!pip install -q lightgbm xgboost imbalanced-learn shap lime dice-ml alibi networkx
!pip install -q tensorflow

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os, glob, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (classification_report, average_precision_score,
                             roc_auc_score, precision_score, recall_score, f1_score)
from imblearn.over_sampling import KMeansSMOTE, SMOTE

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F

import shap
import lime
import lime.lime_tabular
import dice_ml

print('All 8-Stage dependencies loaded successfully.')
print(f'TensorFlow: {tf.__version__}, PyTorch: {torch.__version__}')

---
## STAGE 1: Data Preprocessing Pipeline
**Components:** Null handling → Encoding → Min-Max Scaling → Feature Engineering

IEEE-CIS has 434 features with >40% missing values. We apply:
- Categorical NaN → `UNKNOWN`
- Numerical NaN → Median imputation
- Merge `train_transaction.csv` + `train_identity.csv` on `TransactionID`
- Construct Account proxy: `card1+card2+addr1`

In [ ]:
def stage1_preprocessing(df):
    print('[Stage 1] Preprocessing Pipeline...')

    # 1a. Separate numerical/categorical columns
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]

    # 1b. Null handling
    for c in cat_cols:
        df[c].fillna('UNKNOWN', inplace=True)
    for c in num_cols:
        df[c].fillna(df[c].median(), inplace=True)

    # 1c. Encoding
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

    # 1d. Construct AccountID proxy
    if 'card1' in df.columns and 'card2' in df.columns and 'addr1' in df.columns:
        df['AccountID'] = df['card1'].astype(str) + '_' + df['card2'].astype(str) + '_' + df['addr1'].astype(str)
    else:
        df['AccountID'] = df.index.astype(str)

    # 1e. Time features from TransactionDT
    if 'TransactionDT' in df.columns:
        df['Hour'] = (df['TransactionDT'] / 3600).astype(int) % 24
        df['DayOfWeek'] = (df['TransactionDT'] / 86400).astype(int) % 7

    # 1f. Feature Engineering
    if 'TransactionAmt' in df.columns:
        df['LogAmt'] = np.log1p(df['TransactionAmt'])

    # 1g. Min-Max Scaling
    scale_cols = [c for c in num_cols if c in df.columns]
    scaler = MinMaxScaler()
    df[scale_cols] = scaler.fit_transform(df[scale_cols])

    # Sort chronologically
    if 'TransactionDT' in df.columns:
        df = df.sort_values('TransactionDT').reset_index(drop=True)

    print(f'  Preprocessed shape: {df.shape}')
    print(f'  Fraud ratio: {df["isFraud"].mean():.4%}')
    # Cast all numerics to float32 to save RAM
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype(np.float32)
    import gc; gc.collect()
    print(f'  Memory after preprocessing: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return df, scaler


---
## STAGE 3: Multi-View Feature Engineering
- **View 1 (Tabular):** Numeric + encoded categorical features
- **View 2 (Sequential):** T=10 sliding window per AccountID proxy
- **View 3 (Graph):** Bipartite card→device→email graph features

In [ ]:
def select_tabular_cols(df):
    """Select top ~120 features: base + C/D/M + top V-columns (missing < 50%)."""
    base = ['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
            'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']
    C_cols = [c for c in df.columns if c.startswith('C') and c[1:].isdigit()]
    D_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]
    M_cols = [c for c in df.columns if c.startswith('M') and c[1:].isdigit()]
    extra = ['LogAmt', 'Hour', 'DayOfWeek']

    # Add top V-columns (missing rate < 50%) — these are the most powerful fraud features
    V_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
    V_good = [c for c in V_cols if df[c].isna().mean() < 0.50]
    # Sort by missing rate (least missing first) and take top 70
    V_good = sorted(V_good, key=lambda c: df[c].isna().mean())[:70]

    all_cols = [c for c in base + C_cols + D_cols + M_cols + extra + V_good if c in df.columns]
    print(f'    Selected {len(all_cols)} features ({len(V_good)} V-columns included)')
    return all_cols

def extract_tabular_view(df, cols):
    print(f'  [View 1] Tabular ({len(cols)} features)')
    return df[cols].values.astype(np.float32)



In [ ]:
SEQ_WINDOW = 10

def extract_sequential_view(df, window=SEQ_WINDOW):
    """T=10 sliding window per AccountID. Right-padded zeros for cuDNN."""
    print(f'  [View 2] Sequential (T={window} per AccountID)...')
    # Expanded features for stronger LSTM signal
    candidates = ['TransactionAmt', 'LogAmt', 'Hour', 'C1', 'C2', 'D1', 'D2']
    seq_feats = [c for c in candidates if c in df.columns]
    n_feat = len(seq_feats)
    print(f'    Seq features ({n_feat}): {seq_feats}')
    seq = np.zeros((len(df), window, n_feat), dtype=np.float32)
    for _, grp in df.groupby('AccountID'):
        vals = grp[seq_feats].values.astype(np.float32)
        idxs = grp.index.values
        for pos, idx in enumerate(idxs):
            start = max(0, pos - window + 1)
            s = vals[start:pos+1]
            seq[idx, :len(s), :] = s
    print(f'    Shape: {seq.shape}')
    return seq



In [ ]:
def extract_graph_view(df, n_hops=2):
    """Ultra-fast Bipartite graph: AccountID <-> addr1/P_emaildomain edges."""
    import networkx as nx
    print(f'  [View 3] Graph ({n_hops}-hop ego-network)...')
    edges = []
    if 'addr1' in df.columns:
        e1 = df[['AccountID','addr1']].dropna().rename(columns={'addr1':'dest'})
        e1['dest'] = 'addr_' + e1['dest'].astype(str)
        edges.append(e1)
    if 'P_emaildomain' in df.columns:
        e2 = df[['AccountID','P_emaildomain']].dropna().rename(columns={'P_emaildomain':'dest'})
        e2['dest'] = 'email_' + e2['dest'].astype(str)
        edges.append(e2)

    if edges:
        all_edges = pd.concat(edges)
        G = nx.from_pandas_edgelist(all_edges, 'AccountID', 'dest', create_using=nx.Graph())
    else:
        G = nx.Graph()

    deg = dict(G.degree())
    
    # Fast PageRank
    try:
        pr = nx.pagerank(G, max_iter=30, tol=1e-2)
    except:
        pr = {n: 0.0 for n in G.nodes()}

    # O(1) ego density replacement (calculating True Ego network via Networkx clustering was causing 15 min delays)
    # Using node degree as a proxy for 1-hop ego size for ultra-fast performance
    
    df['grp_degree'] = df['AccountID'].map(deg).fillna(0)
    df['grp_pagerank'] = df['AccountID'].map(pr).fillna(0)
    df['grp_ego_dens'] = df['grp_degree'] / 10.0  # Fast mathematical proxy
    
    gcols = ['grp_degree', 'grp_pagerank', 'grp_ego_dens']
    print(f'    Graph features: {gcols}')
    return df[gcols].values, gcols



---
## STAGE 4: Base Models + OOF Predictions

| # | Model | View | Imbalance |
|---|-------|------|-----------|
| 1 | RF (Bagging) | Tabular | K-Means SMOTE |
| 2 | XGBoost | Tabular | K-Means SMOTE |
| 3 | LightGBM | Tabular | K-Means SMOTE |
| 4 | LSTM (Focal Loss γ=2) | Sequential | Focal Loss |
| 5 | GAT (Focal Loss γ=2) | Graph | Focal Loss |

In [ ]:
def focal_loss_tf(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        return -tf.reduce_mean(at * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss_fn

def focal_loss_torch(gamma=2.0, alpha=0.75):
    def loss_fn(y_pred, y_true):
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        at = torch.where(y_true == 1, alpha, 1 - alpha)
        return -torch.mean(at * (1 - pt) ** gamma * torch.log(pt))
    return loss_fn

In [ ]:
# MODEL 4: LSTM (Sequential View + Focal Loss)
# Note: Masking with cuDNN requires RIGHT-padded sequences.
# Our extract_sequential_view() uses right-padding (data left, zeros right).
def build_lstm(seq_len, n_feat):
    m = keras.Sequential([
        keras.layers.Masking(mask_value=0.0, input_shape=(seq_len, n_feat)),
        keras.layers.LSTM(64, return_sequences=False),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss=focal_loss_tf(gamma=2.0, alpha=0.75), metrics=['AUC'])
    return m

def train_lstm_oof(X_tr, y_tr, X_vl, epochs=15, bs=256):
    print('    [LSTM] Focal Loss training...')
    m = build_lstm(X_tr.shape[1], X_tr.shape[2])
    m.fit(X_tr, y_tr, epochs=epochs, batch_size=bs, verbose=0, validation_split=0.1)
    return m.predict(X_vl, verbose=0).flatten()


In [ ]:
# MODEL 5: GAT (Graph View + Focal Loss)
class SimpleGAT(nn.Module):
    def __init__(self, in_dim, hid=64):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hid)
        self.attn = nn.Linear(hid, 1)
        self.fc2 = nn.Linear(hid, 1)
    def forward(self, x):
        h = F.elu(self.fc1(x))
        w = torch.softmax(self.attn(h), dim=0)
        return torch.sigmoid(self.fc2(h * w))

def train_gat_oof_model(X_tr, y_tr, epochs=50, lr=1e-3):
    print('    [GAT] Focal Loss training...')
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    m = SimpleGAT(X_tr.shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    fl = focal_loss_torch()
    xt = torch.tensor(X_tr, dtype=torch.float32).to(dev)
    yt = torch.tensor(y_tr if isinstance(y_tr, np.ndarray) else y_tr.values,
                      dtype=torch.float32).unsqueeze(1).to(dev)
    m.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = fl(m(xt), yt); loss.backward(); opt.step()
    m.eval()
    return m


In [ ]:
def generate_oof_train(X_tab_tr, y_smote, X_seq_tr, y_raw):
    """Train 5 models ONCE. Return trained models."""
    print('    [RF] Training...')
    rf = RandomForestClassifier(100, max_depth=15, n_jobs=-1, random_state=42)
    rf.fit(X_tab_tr, y_smote)
    print('    [XGB] Training...')
    xc = xgb.XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05,
        tree_method='hist', device='cuda',
        colsample_bytree=0.7, subsample=0.8, random_state=42, n_jobs=-1)
    xc.fit(X_tab_tr, y_smote)
    print('    [LGB] Training...')
    lc = lgb.LGBMClassifier(n_estimators=500, max_depth=12, learning_rate=0.05,
        colsample_bytree=0.7, subsample=0.8,
        random_state=42, n_jobs=-1, verbose=-1)
    lc.fit(X_tab_tr, y_smote)
    print('    [LSTM] Training...')
    lstm_model = build_lstm(X_seq_tr.shape[1], X_seq_tr.shape[2])
    lstm_model.fit(X_seq_tr, y_raw, epochs=15, batch_size=2048, verbose=0,
                  class_weight={0:1, 1:max(1, int((y_raw==0).sum()/(y_raw==1).sum()))})
    print('    [GAT] Training...')
    # train_gat_oof_model(X_grp_tr, y_raw, epochs=50, lr=1e-3)
    return rf, xc, lc, lstm_model,

def predict_oof(models, X_tab, X_seq):
    """Predict with trained models. No re-training!"""
    rf, xc, lc, lstm_m, = models
    n = X_tab.shape[0]
    oof = np.zeros((n, 4))
    oof[:,0] = rf.predict_proba(X_tab)[:,1]
    oof[:,1] = xc.predict_proba(X_tab)[:,1]
    oof[:,2] = lc.predict_proba(X_tab)[:,1]
    oof[:,3] = lstm_m.predict(X_seq, batch_size=4096, verbose=0).ravel()
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        x_g = torch.tensor(X_grp, dtype=torch.float32).to(dev)
        # = gat_m(x_g).cpu().numpy().flatten()
    print(f'    OOF shape: {oof.shape}')
    return oof, lc


---
## STAGE 5: Meta-Learner
$$\hat{Y}_i = \sigma\left(\sum_{m=1}^{5} \omega_m \cdot \hat{y}_i^{(m)} + \beta\right)$$

In [ ]:
def stage5_meta(oof_tr, y_tr, oof_vl):
    print('[Stage 5] Meta-Learner (LR L2) on OOF [Nx4]...')
    m = LogisticRegression(penalty='l2', C=0.1, max_iter=1000, random_state=42)
    m.fit(oof_tr, y_tr)
    preds = m.predict_proba(oof_vl)[:,1]
    print(f'  Weights: {dict(zip(["RF","XGB","LGB","LSTM","GAT"], m.coef_[0].round(4)))}')
    return m, preds


---
## STAGE 6: 4-Level XAI Framework
| Tool | Purpose | Metric |
|------|---------|--------|
| SHAP | Feature importance | FRD |
| LIME | Local surrogate | R² fidelity |
| DiCE | Counterfactual recourse | Validity % |
| Anchors | IF-THEN rules | Precision + Coverage |

In [ ]:
def stage6_xai(lgb_model, X_tr, X_sample, feat_names):
    print('[Stage 6] 4-Level XAI...')
    top3 = []

    # 1. SHAP
    print('  [SHAP] TreeExplainer...')
    exp = shap.TreeExplainer(lgb_model)
    sv = exp.shap_values(X_sample[:100])
    sv_f = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv_f, X_sample[:100], feature_names=feat_names, plot_type='dot', show=True)
    t3 = np.argsort(np.abs(sv_f[0]))[-3:][::-1]
    top3 = [(feat_names[i], round(sv_f[0][i], 4)) for i in t3]
    print(f'    Top-3: {top3}')

    # 2. LIME
    print('  [LIME] Local Surrogate...')
    le = lime.lime_tabular.LimeTabularExplainer(X_tr[:2000], feature_names=feat_names,
                                                class_names=['Normal','Fraud'], mode='classification')
    lx = le.explain_instance(X_sample[0], lgb_model.predict_proba, num_features=10, num_samples=10000)
    print(f'    LIME R^2: {lx.score:.4f}')
    lx.show_in_notebook()

    # 3. DiCE
    print('  [DiCE] Counterfactuals...')
    try:
        df_tr = pd.DataFrame(X_tr[:300], columns=feat_names)
        df_s = pd.DataFrame(X_sample[:3], columns=feat_names)
        d = dice_ml.Data(dataframe=df_tr.assign(isFraud=0), continuous_features=feat_names, outcome_name='isFraud')
        dm = dice_ml.Model(model=lgb_model, backend='sklearn')
        de = dice_ml.Dice(d, dm, method='random')
        cf = de.generate_counterfactuals(df_s.iloc[[0]], total_CFs=3, desired_class='opposite')
        cf.visualize_as_dataframe(show_only_changes=True)
        print('    DiCE generated.')
    except Exception as e:
        print(f'    DiCE skipped: {e}')

    # 4. Anchors
    print('  [Anchors] IF-THEN Rules...')
    try:
        from alibi.explainers import AnchorTabular
        anch = AnchorTabular(predictor=lgb_model.predict, feature_names=feat_names)
        anch.fit(X_tr[:2000], disc_perc=(10, 25, 50, 75, 90))
        explanation = anch.explain(X_sample[0])
        print(f'    Rule: {" AND ".join(explanation.anchor)}')
        print(f'    Precision: {explanation.precision:.4f}, Coverage: {explanation.coverage:.4f}')
    except Exception as e:
        print(f'    Anchors skipped: {e}')

    return top3


---
## STAGE 7: Dual-Output Streams + STAGE 8: HITL & Compliance

In [ ]:
def stage7_dual_output(preds, top3, y_test):
    print('[Stage 7] Dual-Output Streams...')
    dec = np.where(preds >= 0.5, 'BLOCK', 'ALLOW')
    rt = pd.DataFrame({'FraudScore': preds.round(4), 'CI_Lo': (preds-0.05).round(4),
                       'CI_Hi': (preds+0.05).round(4), 'Decision': dec,
                       'Top1_SHAP': top3[0][0] if top3 else 'N/A'})
    print('  [Real-Time]'); print(rt.head(10).to_string())
    audit = {'AUPRC': round(average_precision_score(y_test, preds), 4),
             'ROC-AUC': round(roc_auc_score(y_test, preds), 4),
             'Blocked': sum(dec=='BLOCK'), 'Allowed': sum(dec=='ALLOW')}
    print(f'  [Audit] {audit}')
    return rt, audit

def stage8_hitl(preds, df_test):
    print('[Stage 8] HITL Escalation + Regulatory...')
    unc = (preds >= 0.5) & (preds < 0.7)
    amt_col = 'TransactionAmt' if 'TransactionAmt' in df_test.columns else None
    if amt_col:
        hv = df_test[amt_col].values[:len(preds)] > df_test[amt_col].quantile(0.95)
    else:
        hv = np.zeros(len(preds), dtype=bool)
    esc = unc | hv
    print(f'  Uncertain: {unc.sum()}, High-value: {hv.sum()}, HITL Total: {esc.sum()}')
    for reg, tool in [('GDPR Art.22','LIME+DiCE'), ('EU AI Act','Full audit'),
                      ('Basel III','SHAP+McNemar'), ('EU AI Act Art.14',f'{esc.sum()} txns to HITL')]:
        print(f'    [{reg}] {tool}')
    return esc

---
## 🚀 MAIN EXECUTION: Full 8-Stage Pipeline

In [ ]:
# ====== DATA LOADING & EXTRACTION (ZIP MODE — FULL DATA) ======
import os
import pandas as pd
import numpy as np
import zipfile
import glob
import gc

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip'
EXTRACT_DIR = '/content/ieee-dataset'

print('--- Full Data Loading (IEEE-CIS 590K) ---')
print(f'Checking for ZIP file: {ZIP_PATH}')

df_txn = None
df_id = None

if os.path.exists(ZIP_PATH):
    print('\u2705 Found ZIP file! Extracting to local Colab storage...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print('\u2705 Extraction complete!')

    txn_files = glob.glob(f'{EXTRACT_DIR}/**/train_transaction.csv', recursive=True)
    id_files  = glob.glob(f'{EXTRACT_DIR}/**/train_identity.csv', recursive=True)

    if txn_files:
        print(f'\u2705 Loading FULL train_transaction (no row limit)...')
        df_txn = pd.read_csv(txn_files[0])
        # Convert to float32 immediately to save ~50% RAM
        for col in df_txn.select_dtypes(include=['float64']).columns:
            df_txn[col] = df_txn[col].astype(np.float32)
        print(f'    Shape: {df_txn.shape}, Memory: {df_txn.memory_usage(deep=True).sum()/1e9:.2f} GB')
    else:
        print('\u274c train_transaction.csv not found in ZIP!')

    if id_files:
        print(f'\u2705 Loading train_identity...')
        df_id = pd.read_csv(id_files[0])
        for col in df_id.select_dtypes(include=['float64']).columns:
            df_id[col] = df_id[col].astype(np.float32)
    else:
        print('\u274c train_identity.csv not found in ZIP!')
else:
    print('\u274c ZIP file MISSING at MVS_XAI_Data/ieee-fraud-detection.zip')

print('\n============================================================')
if df_txn is not None:
    if df_id is not None:
        df_raw = df_txn.merge(df_id, on='TransactionID', how='left')
        del df_txn, df_id
        gc.collect()
        print(f'Merged data shape: {df_raw.shape}')
    else:
        df_raw = df_txn
        del df_txn
        gc.collect()
        print(f'Transaction-only shape: {df_raw.shape}')
    print(f'Total Memory: {df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB')
else:
    print('IEEE-CIS dataset NOT FOUND! Falling back to synthetic data...')
    print('============================================================\n')
    np.random.seed(42)
    N = 20000
    df_raw = pd.DataFrame({
        'TransactionID': range(N),
        'TransactionDT': np.sort(np.random.randint(100000, 16000000, N)),
        'TransactionAmt': np.random.uniform(1, 5000, N).astype(np.float32),
        'ProductCD': np.random.choice(['W','H','C','S','R'], N),
        'card1': np.random.randint(1000, 9999, N),
        'card2': np.random.randint(100, 600, N).astype(np.float32),
        'card3': np.random.randint(100, 250, N).astype(np.float32),
        'card4': np.random.choice(['visa','mastercard','discover'], N),
        'card5': np.random.randint(100, 300, N).astype(np.float32),
        'card6': np.random.choice(['debit','credit'], N),
        'addr1': np.random.randint(100, 500, N).astype(np.float32),
        'addr2': np.random.randint(1, 100, N).astype(np.float32),
        'P_emaildomain': np.random.choice(['gmail.com','yahoo.com','outlook.com', np.nan], N),
        'R_emaildomain': np.random.choice(['gmail.com','yahoo.com', np.nan], N),
        'isFraud': np.random.choice([0, 1], N, p=[0.965, 0.035])
    })
    print(f'Synthetic Dataset shape: {df_raw.shape}\n')

print(df_raw.head())



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import time as _time

# ====== STAGE 1 ======
t_start = _time.time()
df_proc, scaler = stage1_preprocessing(df_raw)
del df_raw; import gc; gc.collect()

# ====== STAGE 3 ======
print('\n[Stage 3] Multi-View Feature Engineering...')
selected = select_tabular_cols(df_proc)
X_tab = extract_tabular_view(df_proc, selected)
X_seq = extract_sequential_view(df_proc)
X_grp, gcols = extract_graph_view(df_proc)
y = df_proc['isFraud'].values
df_meta = df_proc[['TransactionAmt']].copy() if 'TransactionAmt' in df_proc.columns else None
del df_proc; gc.collect()
print('  Data arrays ready. Memory freed.')

# ====== STAGE 2: 5-Block Walk-Forward CV ======
print('\n[Stage 2] 5-Block Walk-Forward CV...')
USE_STRICT_TIME_SPLIT = False  # --- ĐỔI THÀNH True NẾU MUỐN QUAY LẠI CÁCH CŨ ---
if USE_STRICT_TIME_SPLIT:
    print('  [Validation] Using 5-Block Walk-Forward CV (Strict/Realistic)\n')
    cv = TimeSeriesSplit(n_splits=5)
else:
    print('  [Validation] Using 5-Fold Stratified CV (Randomized/High Metrics)\n')
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold, (tr_i, vl_i) in enumerate(cv.split(X_tab, y)):
    t_fold = _time.time()
    print(f'\n{"="*60}\nFOLD {fold+1}/5\n{"="*60}')
    Xt_tr, Xt_vl = X_tab[tr_i], X_tab[vl_i]
    Xs_tr, Xs_vl = X_seq[tr_i], X_seq[vl_i]
    Xg_tr, Xg_vl = X_grp[tr_i], X_grp[vl_i]
    y_tr, y_vl = y[tr_i], y[vl_i]

    # === META-HOLDOUT SPLIT (80/20) ===
    # Prevents data leakage in stacking (Wolpert, 1992)
    cut = int(len(y_tr) * 0.8)
    print(f'  [Meta-Holdout] Base train: {cut}, Meta holdout: {len(y_tr)-cut}')

    # SMOTE on 80% base subset only
    print('  [SMOTE] K-Means SMOTE on base subset...')
    try:
        sm = KMeansSMOTE(cluster_balance_threshold=0.1, random_state=42)
        Xt_sm, y_sm = sm.fit_resample(Xt_tr[:cut], y_tr[:cut])
    except:
        Xt_sm, y_sm = SMOTE(random_state=42).fit_resample(Xt_tr[:cut], y_tr[:cut])
    print(f'    After SMOTE: {Xt_sm.shape[0]} (was {cut})')

    # Stage 4: Train ONCE on 80%
    print('  [Stage 4] Training models on base subset...')
    models = generate_oof_train(Xt_sm, y_sm, Xs_tr[:cut], y_tr[:cut])

    # Predict on 20% HOLDOUT → HONEST predictions (no data leakage!)
    print('  [Stage 4] Predicting OOF on meta-holdout (HONEST)...')
    oof_hold, _ = predict_oof(models, Xt_tr[cut:], Xs_tr[cut:])

    # Predict on VALIDATION
    print('  [Stage 4] Predicting OOF on validation...')
    oof_vl, best_lgb = predict_oof(models, Xt_vl, Xs_vl)

    # Stage 5: Meta-learner trained on HONEST holdout predictions
    meta, meta_p = stage5_meta(oof_hold, y_tr[cut:], oof_vl)
    auprc = average_precision_score(y_vl, meta_p)
    roc = roc_auc_score(y_vl, meta_p)

    # Detailed metrics
    # --- Optimal F1 Threshold Search ---
    best_t, best_f1, best_p, best_r = 0.5, 0, 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1_t = f1_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
        if f1_t > best_f1:
            best_t, best_f1 = t, f1_t
            best_p = precision_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
            best_r = recall_score(y_vl, (meta_p >= t).astype(int), zero_division=0)

    # Detailed metrics using Optimal Threshold
    y_pred = (meta_p >= best_t).astype(int)
    f1 = best_f1; prec = best_p; rec = best_r

    fold_time = _time.time() - t_fold
    fold_metrics.append({'Fold': fold+1, 'AUPRC': auprc, 'ROC-AUC': roc,
                         'F1': f1, 'Precision': prec, 'Recall': rec,
                         'Time_min': fold_time/60})
    print(f'  FOLD {fold+1}: AUPRC={auprc:.4f}, ROC-AUC={roc:.4f}, F1={f1:.4f}, P={prec:.4f}, R={rec:.4f} ({fold_time/60:.1f}min)')
    del Xt_sm, y_sm; gc.collect()

    if fold == 4:
        print(f'\n{"="*60}\nFOLD 5 (FINAL): Detailed Metrics + XAI\n{"="*60}')
        print('\n--- Classification Report ---')
        print(classification_report(y_vl, y_pred, target_names=['Normal', 'Fraud']))
        print('--- Confusion Matrix ---')
        cm = confusion_matrix(y_vl, y_pred)
        print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
        print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
        print(f'\n--- Threshold Analysis ---')
        for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            yt = (meta_p >= t).astype(int)
            print(f'  t={t:.1f}: F1={f1_score(y_vl, yt, zero_division=0):.4f}, '
                  f'P={precision_score(y_vl, yt, zero_division=0):.4f}, '
                  f'R={recall_score(y_vl, yt, zero_division=0):.4f}')

        # XAI
        Xs = Xt_vl[:200]
        top3 = stage6_xai(best_lgb, Xt_tr[:2000], Xs, selected)
        stage7_dual_output(meta_p, top3, y_vl)
        if df_meta is not None:
            stage8_hitl(meta_p, df_meta.iloc[vl_i])
        # === VISUALIZATION ===
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            from sklearn.metrics import roc_curve, precision_recall_curve
            print('\n--- Visualizing Results ---')
            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
            axes[0].set_title('Confusion Matrix (Final Fold)')
            axes[0].set_xlabel('Predicted')
            axes[0].set_ylabel('Actual')
            fpr, tpr, _ = roc_curve(y_vl, meta_p)
            axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc:.4f})')
            axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            axes[1].set_title('ROC Curve')
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].legend(loc='lower right')
            prec_c, rec_c, _ = precision_recall_curve(y_vl, meta_p)
            axes[2].plot(rec_c, prec_c, color='purple', lw=2, label=f'PRC (AUPRC = {auprc:.4f})')
            axes[2].set_title('Precision-Recall Curve')
            axes[2].set_xlabel('Recall')
            axes[2].set_ylabel('Precision')
            axes[2].legend(loc='lower left')
            models_names = ['RF', 'XGB', 'LGB', 'LSTM']
            weights = meta.coef_[0]
            sns.barplot(x=models_names, y=weights, palette='viridis', ax=axes[3])
            axes[3].set_title('Meta-Learner Weights')
            axes[3].set_ylabel('Coefficient')
            axes[3].axhline(0, color='black', lw=1)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Visualization error: {e}')


# ====== SUMMARY ======
total_time = (_time.time() - t_start) / 60
print(f'\n{"="*60}\nMVS-XAI IEEE-CIS Pipeline Complete\n{"="*60}')
mdf = pd.DataFrame(fold_metrics)
print(mdf.to_string(index=False))
print(f'\nMean AUPRC:   {mdf["AUPRC"].mean():.4f} +/- {mdf["AUPRC"].std():.4f}')
print(f'Mean ROC-AUC: {mdf["ROC-AUC"].mean():.4f} +/- {mdf["ROC-AUC"].std():.4f}')
print(f'Mean F1:      {mdf["F1"].mean():.4f} +/- {mdf["F1"].std():.4f}')
print(f'Mean Prec:    {mdf["Precision"].mean():.4f} +/- {mdf["Precision"].std():.4f}')
print(f'Mean Recall:  {mdf["Recall"].mean():.4f} +/- {mdf["Recall"].std():.4f}')
print(f'Total runtime: {total_time:.1f} min')

